# HelioGuard Phase 6: SFT warm start (Colab T4)

Run the cells in order. The cell marked **Resume** is the same training command: run it again after a disconnect and training continues from the last checkpoint on Drive.

Setup once: Colab secret `GITHUB_TOKEN` (only needed if the repo is private) and a GPU runtime (T4). Nothing secret is stored in this notebook.

In [2]:
# 1. Clone the repo (token from the Colab secret, never printed or saved in the remote URL)
import os, subprocess
from google.colab import userdata
REPO = "https://github.com/nithinraj49/helioguard"
if not os.path.exists("/content/helioguard"):
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None   # public repo: clone without a token
    url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
    subprocess.run(["git", "clone", "-q", url, "/content/helioguard"], check=True)
    subprocess.run(["git", "-C", "/content/helioguard", "remote", "set-url", "origin", REPO + ".git"], check=True)
%cd /content/helioguard
!git log --oneline -1

/content/helioguard
1d86d87 (HEAD -> main, origin/main, origin/HEAD) Phase 6: Amendment 2 - compact view carries true min altitude and hour; traces regenerated (assistant turns and rewards identical); trainer config field; sft_eval --greedy-only


In [ ]:
# 2. Install (Unsloth first, then the project)
!pip -q install unsloth
!pip -q install -e ".[sft]" ortools

In [ ]:
# 3. Mount Drive: checkpoints and the final adapter live here, so they survive disconnects
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints"
OUT = "/content/drive/MyDrive/helioguard/sft_1p5b"
os.makedirs(CKPT, exist_ok=True)

In [ ]:
# 4. Regenerate the SFT traces (deterministic, CPU, train tasks only) and show the data card
!python scripts/generate_sft_traces.py --workers 2 | head -60

In [ ]:
# 5. Train (Resume: run this same cell again after a disconnect)
!python scripts/train_sft.py configs/sft_1p5b.json --checkpoint-dir "$CKPT" --out-dir "$OUT"

In [ ]:
# 6. Evaluate the adapter on the 50 held-out tasks (greedy + temperature 0.7 x 3 seeds, bootstrap CIs, gates)
!python -m helioguard.sft_eval --model Qwen/Qwen2.5-1.5B-Instruct --adapter "$OUT/adapter" --out "$OUT/eval" --tag sft_1p5b

Copy `sft_1p5b_eval_summary.json` and `train_summary.json` from Drive back into `results/phase6/` in the repo to commit them. Do not commit checkpoints or adapters unless asked.

In [3]:
!git clone https://github.com/nithinraj49/helioguard.git /content/helioguard
%cd /content/helioguard
!git log --oneline -1

fatal: destination path '/content/helioguard' already exists and is not an empty directory.
/content/helioguard
1d86d87 (HEAD -> main, origin/main, origin/HEAD) Phase 6: Amendment 2 - compact view carries true min altitude and hour; traces regenerated (assistant turns and rewards identical); trainer config field; sft_eval --greedy-only


In [4]:
%cd /content/helioguard
!pip install -q -e .
!pip install -q peft accelerate

/content/helioguard
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for helioguard (pyproject.toml) ... done


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
CKPT = "/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints"
OUT = "/content/drive/MyDrive/helioguard/sft_1p5b"
os.makedirs(CKPT, exist_ok=True)
print(CKPT, OUT)

/content/drive/MyDrive/helioguard/sft_1p5b/checkpoints /content/drive/MyDrive/helioguard/sft_1p5b


In [7]:
%cd /content/helioguard
!python scripts/generate_sft_traces.py --workers 2 > /content/trace_gen.log 2>&1
!tail -5 /content/trace_gen.log
!wc -l data/sft/train_traces.jsonl
!cp data/sft/train_traces.jsonl /content/drive/MyDrive/helioguard/

/content/helioguard
    "mean": 3392,
    "max": 3595,
    "min": 2864
  }
}
800 data/sft/train_traces.jsonl


In [8]:
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv
!cat /content/drive/MyDrive/helioguard/sft_1p5b_smoke/train_summary.json
!cat configs/sft_1p5b.json

1d86d87 (HEAD -> main, origin/main, origin/HEAD) Phase 6: Amendment 2 - compact view carries true min altitude and hour; traces regenerated (assistant turns and rewards identical); trainer config field; sft_eval --greedy-only
name, memory.total [MiB]
Tesla T4, 15360 MiB
{
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "trainer": "transformers+peft",
  "training_seed": 0,
  "steps_done": 20,
  "steps_total": 20,
  "resumed_from": 0,
  "examples": 800,
  "skipped_too_long": 0,
  "steps_per_epoch": 100,
  "final_loss": 0.007699005771428347,
  "first_loss": 0.11965257953852415,
  "config": {
    "model": "Qwen/Qwen2.5-1.5B-Instruct",
    "data": "data/sft/train_traces.jsonl",
    "out_dir": "/content/drive/MyDrive/helioguard/sft_1p5b_smoke",
    "checkpoint_dir": "/content/drive/MyDrive/helioguard/sft_1p5b_smoke/checkpoints",
    "max_seq_len": 4096,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.0,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o

In [12]:
SMOKE2 = "/content/drive/MyDrive/helioguard/sft_1p5b_smoke2"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True timeout 960 python scripts/train_sft.py configs/sft_1p5b.json --checkpoint-dir "$SMOKE2/checkpoints" --out-dir "$SMOKE2" --max-steps 20
!ls -la "$SMOKE2/checkpoints"

Loading weights: 100% 338/338 [00:02<00:00, 144.07it/s]
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/content/helioguard/verispace_engine/sft.py:264: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  loss_sum += float(loss)
{"step": 1, "loss": 0.11965257953852415, "grad_norm": 0.5901414752006531, "lr": 2e-05, "label_tokens": 2913, "elapsed_s": 54.6}
{"step": 2, "loss": 0.11542756203562021, "grad_norm": 0.5297238826751709, "lr": 4e-05, "label_tokens": 2915, "elapsed_s": 113.0}
{"step": 3, "loss": 0.0773198651149869, "grad_norm": 0.34934601187705994, "lr": 6.000000000000001e-05, "label_tokens": 2710, "elapsed_s": 166.5}
{"step": 4, "loss": 0.06205701408907771, "grad_norm": 0.3237033784389496, "lr": 8e-05, "label_tokens": 2906, "elapsed_s": 223

In [13]:
SMOKE2 = "/content/drive/MyDrive/helioguard/sft_1p5b_smoke2"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_sft.py configs/sft_1p5b.json --checkpoint-dir "$SMOKE2/checkpoints" --out-dir "$SMOKE2" --max-steps 20

Loading weights: 100% 338/338 [00:02<00:00, 125.45it/s]
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/content/helioguard/verispace_engine/sft.py:264: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  loss_sum += float(loss)
{"step": 12, "loss": 0.012901814188808203, "grad_norm": 0.12834720313549042, "lr": 0.00018, "label_tokens": 2757, "elapsed_s": 51.9}
{"step": 13, "loss": 0.012377112114336342, "grad_norm": 0.27124977111816406, "lr": 0.00016, "label_tokens": 2744, "elapsed_s": 106.3}
{"step": 14, "loss": 0.010264301556162536, "grad_norm": 0.14169833064079285, "lr": 0.00014, "label_tokens": 2791, "elapsed_s": 161.4}
{"step": 15, "loss": 0.008904295216780156, "grad_norm": 0.10318493098020554, "lr": 0.00012, "label_tokens": 2871, "elapsed_s":